In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
dataset=pd.read_csv("/content/Synthetic.csv")

In [ ]:
dataset.head()

In [ ]:
processed_data=dataset.drop(columns=["Unnamed: 0"])

In [ ]:
processed_data.head()

In [ ]:
processed_data['X'].values

In [ ]:
processed_data['X'].values.shape

BATCH GRADIENT

In [ ]:
class BatchGradient:
 def __init__(self):
    self.learning_rate = 0.0001
    self.epochs = 10000
    self.tolerance = 0.001
    self.epoch = 1
    self.loss = 1
    self.loss_history = []
 def preprocess(self):
    self.feature_values=processed_data.iloc[:,0].values
    self.target_values=processed_data.iloc[:,1].values
    return (self.feature_values,self.target_values)
 def reshape(self):
    self.feature_values = self.feature_values.reshape(1, -1)
    self.target_values = self.target_values.reshape(-1, 1)
    return (self.feature_values,self.target_values)
 def standardize(self):
     self.feature_values = (self.feature_values - np.mean(self.feature_values)) / np.std(self.feature_values)
 def initialise_parameters(self):
    self.intercept = np.array(np.random.randn()).reshape(1,1)
    self.slope = np.array(np.random.randn()).reshape(1,1)
 def calculate(self):
    batch_size_value=50
    while self.epoch <= self.epochs and self.loss > self.tolerance:

     # Forward pass
      self.predicted_values = self.intercept * self.feature_values + self.slope

    # Loss calculation
      self.loss = 1/2 * np.mean((self.predicted_values.T - self.target_values)**2)

    # derivative calculation
      self.d_theta_0 = 1/batch_size_value * np.dot(self.feature_values, (self.predicted_values.T - self.target_values))
      self.d_theta_1 = 1/batch_size_value * np.sum(self.predicted_values.T - self.target_values)

    # Update parameters
      self.intercept = self.intercept - self.learning_rate * self.d_theta_0
      self.slope = self.slope - self.learning_rate * self.d_theta_1

      self.loss_history.append(self.loss)

      self.epoch += 1
    return (self.intercept,self.slope,self.loss_history)

In [ ]:
if __name__ == "__main__":
    optimizer=BatchGradient()
    optimizer.preprocess()
    optimizer.reshape()
    optimizer.standardize()
    optimizer.initialise_parameters()
    (intercept,slope,loss_history)=optimizer.calculate()

In [ ]:
print(intercept)
print(slope)


In [ ]:
plt.plot(loss_history)
plt.xlabel("Number of epochs")
plt.ylabel("Loss (MSE)")

STOCHASTIC GRADIENT DESCENT

In [ ]:
class StochasticGD:
 def __init__(self):
    self.learning_rate = 0.0001
    self.epochs = 10000
    self.tolerance = 0.001
    self.epoch = 1
    self.loss = 1
 def preprocess(self):
    self.feature_values=processed_data.iloc[:,0].values
    self.target_values=processed_data.iloc[:,1].values
    return (self.feature_values,self.target_values)
 def standardize(self):
     self.feature_values = (self.feature_values - np.mean(self.feature_values)) / np.std(self.feature_values)
 def initialise_parameters(self):
    self.intercept = np.array(np.random.randn())
    self.slope = np.array(np.random.randn())
 def calculate(
 self):
    self.loss_history=[]
    while self.epoch <= self.epochs and self.loss > self.tolerance:
         total_epoch_loss=0
         for i in range(self.feature_values.shape[0]):
           sample_index=np.random.randint(0,self.feature_values.shape[0])
           self.predicted_values=self.feature_values[sample_index]*self.slope+self.intercept
           self.loss = 1/2*(self.predicted_values - self.target_values[sample_index])**2
    # derivative calculation
           self.d_theta_1=  self.feature_values[sample_index]*(self.predicted_values - self.target_values[sample_index])
           self.d_theta_0 = (self.predicted_values - self.target_values[sample_index])
           self.intercept = self.intercept - self.learning_rate * self.d_theta_0
           self.slope = self.slope - self.learning_rate * self.d_theta_1
           total_epoch_loss+=self.loss
         self.epoch += 1
         avg_loss = total_epoch_loss / self.feature_values.shape[0]
         self.loss_history.append(avg_loss)
    return self.intercept,self.slope,self.loss_history

In [ ]:
if __name__ == "__main__":
    optimizer=StochasticGD()
    optimizer.preprocess()
    optimizer.standardize()
    optimizer.initialise_parameters()
    (intercept,slope,loss_history)=optimizer.calculate()

In [ ]:
print(intercept)
print(slope)

In [ ]:
plt.plot(loss_history)
plt.xlabel("Number of epochs")
plt.ylabel("Loss (MSE)")

MINI BATCH GRADIENT DESCENT

In [ ]:
class MiniBatchGD:
 def __init__(self,batch_size):
    self.learning_rate = 0.0001
    self.epochs = 10000
    self.tolerance = 0.001
    self.epoch = 1
    self.loss = 1
    self.batch_size=batch_size
 def preprocess(self):
    self.feature_values=processed_data.iloc[:,0].values
    self.target_values=processed_data.iloc[:,1].values
    return (self.feature_values,self.target_values)
 def standardize(self):
     self.feature_values = (self.feature_values - np.mean(self.feature_values)) / np.std(self.feature_values)
 def initialise_parameters(self):
    self.intercept = np.array(np.random.randn())
    self.slope = np.array(np.random.randn())
 def calculate(self):
    batch_size_value=self.batch_size
    self.loss_history=[]
    while self.epoch <= self.epochs and self.loss > self.tolerance:
        total_epoch_loss=0
        batch_count=1

        for i in range(int(self.feature_values.shape[0]/self.batch_size)):
          sample_index = np.random.choice(self.feature_values.shape[0], self.batch_size, replace=False)
          self.predicted_values=self.feature_values[sample_index]*self.slope+self.intercept

    # Loss calculation
          current_batch_loss = 1/2 * np.mean((self.predicted_values.T- self.target_values[sample_index])**2)
          batch_count+=1
          total_epoch_loss+=current_batch_loss
    # derivative calculation
          self.d_theta_1 = 1/batch_size_value * np.dot(self.feature_values[sample_index], (self.predicted_values.T - self.target_values[sample_index]))
          self.d_theta_0 = 1/batch_size_value * np.sum(self.predicted_values.T - self.target_values[sample_index])

    # Update parameters
          self.intercept = self.intercept - self.learning_rate * self.d_theta_0
          self.slope = self.slope - self.learning_rate * self.d_theta_1

        self.loss= total_epoch_loss/batch_count
        self.loss_history.append(self.loss)
        self.epoch += 1
    return (self.intercept,self.slope,self.loss_history)

In [ ]:
if __name__ == "__main__":
    optimizer=MiniBatchGD(batch_size=10)
    optimizer.preprocess()
    optimizer.standardize()
    optimizer.initialise_parameters()
    (intercept,slope,loss_history)=optimizer.calculate()

In [ ]:
print(intercept)
print(slope)

In [ ]:
plt.plot(loss_history)
plt.xlabel("Number of epochs")
plt.ylabel("Loss (MSE)")